# Kroko streaming ASR with sherpa-onnx

In this notebook I test the Kroko English streaming model with sherpa-onnx.
I compare it with whisper-small int8 on the same podcast (`podcast-offline-test.ipynb`).

- sherpa-onnx install (CPU wheels): https://k2-fsa.github.io/sherpa/onnx/python/install.html#method-1-from-pre-compiled-wheels-cpu-only
- Kroko models: https://huggingface.co/Banafo/Kroko-ASR
- On Windows, install `sherpa-onnx-core` too. Without it, an old `onnxruntime.dll` from System32 can be loaded and the kernel crashes.

In [1]:
import re
import time
from importlib.metadata import version
from pathlib import Path

import jiwer
import numpy as np
import sherpa_onnx
from faster_whisper import decode_audio
from huggingface_hub import snapshot_download
from transformers import WhisperTokenizer

SR = 16000
print("sherpa-onnx version:", version("sherpa-onnx"))

c:\Users\Unal\Desktop\DeepLearningAndOptimization\HES-Hackathon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sherpa-onnx version: 1.13.8


In [2]:
help(sherpa_onnx)

Help on package sherpa_onnx:

NAME
    sherpa_onnx

PACKAGE CONTENTS
    __main__
    _info
    cli
    display
    keyword_spotter
    offline_recognizer
    online_recognizer
    utils

SUBMODULES
    lib

FUNCTIONS
    write_wave(...) method of builtins.pybind11_detail_function_record_v1_msvc_mt_mscver1944 instance
        write_wave(filename: str, samples: collections.abc.Sequence[typing.SupportsFloat], sample_rate: typing.SupportsInt) -> bool


        Write audio samples to a WAV file.

        Args:
          filename:
            Path to the output WAV file.
          samples:
            A 1-D float32 array of audio samples.
          sample_rate:
            The sample rate of the audio.

        Returns:
          True if the file was written successfully.

DATA
    git_date = 'Thu Sep 10 13:52:45 2026'
    git_sha1 = '11afbd00'
    onnxruntime_version = '1.28.2'
    version = '1.13.8'

VERSION
    1.13.8

FILE
    c:\users\unal\desktop\deeplearningandoptimization\hes-hackat

In [3]:
help(sherpa_onnx.online_recognizer)

Help on module sherpa_onnx.online_recognizer in sherpa_onnx:

NAME
    sherpa_onnx.online_recognizer - # Copyright (c)  2023  Xiaomi Corporation

CLASSES
    builtins.object
        OnlineRecognizer

    class OnlineRecognizer(builtins.object)
     |  A class for streaming (online) speech recognition.
     |
     |  It supports multiple model families via factory methods:
     |
     |  - :meth:`from_transducer` -- Zipformer, Nemotron, etc.
     |  - :meth:`from_paraformer` -- Streaming Paraformer
     |  - :meth:`from_zipformer2_ctc` -- Zipformer2 CTC
     |  - :meth:`from_wenet_ctc` -- WeNet CTC
     |
     |  Example using streaming Zipformer transducer::
     |
     |      import numpy as np
     |      import sherpa_onnx
     |
     |      recognizer = sherpa_onnx.OnlineRecognizer.from_transducer(
     |          tokens="./model/tokens.txt",
     |          encoder="./model/encoder-epoch-99-avg-1.int8.onnx",
     |          decoder="./model/decoder-epoch-99-avg-1.onnx",
     |    

## Step 1: Load the model

The model is not inside the sherpa-onnx package. `snapshot_download` downloads it from Hugging Face (only the first time, then it uses the cache).
The model folder has three ONNX files and a token list:
- `encoder.onnx`: Zipformer encoder (the audio part)
- `decoder.onnx`: the predictor (looks at the last 2 tokens)
- `joiner.onnx`: combines one encoder frame and the predictor output, and gives blank or a token
- `tokens.txt`: the vocabulary (650 tokens)

`model_type="zipformer2"` tells sherpa-onnx how to run these files.
With `**decoding` I can give other settings later, for example `decoding_method="modified_beam_search"`.

In [4]:
KROKO_REPO = "csukuangfj/sherpa-onnx-streaming-zipformer-en-kroko-2025-08-06"
model_dir = Path(snapshot_download(KROKO_REPO))


def load_kroko(**decoding) -> sherpa_onnx.OnlineRecognizer:
    settings = dict(decoding_method="greedy_search") | decoding
    return sherpa_onnx.OnlineRecognizer.from_transducer(
        tokens=str(model_dir / "tokens.txt"),
        encoder=str(model_dir / "encoder.onnx"),
        decoder=str(model_dir / "decoder.onnx"),
        joiner=str(model_dir / "joiner.onnx"),
        num_threads=4,
        sample_rate=SR,
        feature_dim=80,
        model_type="zipformer2",
        **settings,
    )


recognizer = load_kroko()

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 900.98it/s]


## Step 2: Streaming function

This function gives the audio to the model in small chunks, like live audio from the loopback.
- `accept_waveform`: adds a chunk to the stream. The model computes the fbank features.
- `is_ready` / `decode_stream`: the encoder runs when there are enough frames (1.28 s of audio).
- At the end I add 0.66 s of silence (tail padding), so the model can finish the last words.

In [5]:
def transcribe_stream(recognizer, samples, chunk_sec=0.1):
    stream = recognizer.create_stream()
    chunk = int(chunk_sec * SR)

    for start in range(0, len(samples), chunk):
        stream.accept_waveform(SR, samples[start : start + chunk])
        while recognizer.is_ready(stream):
            recognizer.decode_stream(stream)

    # tail padding: some silence so the model can finish the last words
    stream.accept_waveform(SR, np.zeros(int(0.66 * SR), dtype=np.float32))
    stream.input_finished()
    while recognizer.is_ready(stream):
        recognizer.decode_stream(stream)

    return recognizer.get_result(stream)

## Step 3: Quick test

The model folder has a short test file. If this works, the setup is correct.

In [6]:
samples = decode_audio(str(model_dir / "test_wavs" / "0.wav"), sampling_rate=SR)
duration = len(samples) / SR

t0 = time.perf_counter()
text = transcribe_stream(recognizer, samples)
elapsed = time.perf_counter() - t0

print(f"{duration:.1f} s audio, RTF {elapsed / duration:.3f}")
print(text)

3.8 s audio, RTF 0.125
Ask not what your country can do for you. Ask what you can do for your country


## Step 4: Podcast test

Same audio and same reference as in `podcast-offline-test.ipynb`, so I can compare with whisper-small int8 (WER 7.3 %, RTF 0.47).
- skip the first 25 s (ad)
- clean the reference: remove sound notes like (SOUNDBITE OF MUSIC) and speaker names
- normalise both texts with the Whisper English normaliser (it also removes notes like (Laughter) and changes number words to digits)
- remove the three ad blocks (long "extra" word blocks) before the final WER

In [7]:
SAMPLES = Path("../samples")
START = 25  # skip the ad at the start (seconds)

audio = decode_audio(str(SAMPLES / "planet_money_1000.mp3"), sampling_rate=SR)
episode = audio[START * SR :]
duration = len(episode) / SR

transcript = (SAMPLES / "transcript_planet_money_1000.txt").read_text(encoding="utf-8")
reference = re.sub(r"^\(.*\)\s*$", "", transcript, flags=re.M)             # sound notes
reference = re.sub(r"^[A-Z][A-Z0-9 .,'#&-]*:", "", reference, flags=re.M)  # speaker names

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-large-v3-turbo")
ref_norm = tokenizer.normalize(reference)
print(f"audio {duration:.0f} s, reference {len(ref_norm.split())} words")

audio 1636 s, reference 4435 words


In [8]:
def podcast_wer(hypothesis):
    out = jiwer.process_words(ref_norm, tokenizer.normalize(hypothesis))

    # the long "extra" blocks are ads, not part of the episode
    ads = [c for c in out.alignments[0]
           if c.type == "insert" and c.hyp_end_idx - c.hyp_start_idx >= 40]
    hyp_words = out.hypotheses[0]
    kept = [w for i, w in enumerate(hyp_words)
            if not any(c.hyp_start_idx <= i < c.hyp_end_idx for c in ads)]
    return jiwer.process_words(ref_norm, " ".join(kept))


t0 = time.perf_counter()
hypothesis = transcribe_stream(recognizer, episode)
elapsed = time.perf_counter() - t0

o = podcast_wer(hypothesis)
print(f"took {elapsed:.0f} s, RTF {elapsed / duration:.3f}")
print(f"WER {100 * o.wer:.1f} %")
print(f"wrong {o.substitutions}, missing {o.deletions}, extra {o.insertions}")
print(hypothesis[:500])

took 114 s, RTF 0.070
WER 8.6 %
wrong 184, missing 137, extra 61
This is Planet Money from NPR. Yeah, it's our one thousandth episode, and to be honest, we spent a lot of time thinking about ways to avoid doing this episode. There's a lot going on right now. So at one point we were like What have we just skip episode one thousand? Go from episode nine ninety nine to episode one thousand and one, and just do the big one thousandth later when we can throw a real party. For my personal suggestion, we could suddenly switch to decimal system, you know, like sudden


## Results (last run)

| Test | WER | RTF | Notes |
|---|---|---|---|
| test_wavs/0.wav (3.8 s) | - | 0.125 | correct text; first call (warm-up), so the RTF is not reliable |
| Podcast, Kroko greedy | 8.6 % | 0.070 | 1636 s in 114 s (earlier run: 0.064); wrong 184, missing 137, extra 61 (reference: 4435 words) |
| Podcast, whisper-small int8 | 7.3 % | 0.47 | from `podcast-offline-test.ipynb` |

- Kroko is about 7 times faster than Whisper on CPU, and the WER is 1.3 points higher.
- Numbers come as words ("nine ninety nine"). The normaliser changes them to digits for the WER, but for claim checking we need inverse text normalization (ITN) later.
- Many missing words: check if this also happens in live streaming.